In [ ]:
"""
Run top-to-bottom on the engineered Excel file (kse30_engineered.xlsx)
Produces model results, plots, and CSV summaries.
"""

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Models & metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, f1_score, confusion_matrix, roc_auc_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

# XGBoost
try:
    from xgboost import XGBRegressor, XGBClassifier
except Exception:
    raise ImportError("Install xgboost: pip install xgboost")

# SHAP (optional: for interpretability)
try:
    import shap
    _has_shap = True
except Exception:
    _has_shap = False

: 

In [2]:
# -------------------------
# Configuration
# -------------------------
INPUT_FILE = "kse30_daily_data_engineered.xlsx"   # change if needed
OUTPUT_DIR = "fyp_outputs"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
# Feature columns: edit if your column names differ
FEATURE_COLUMNS = [
    #existing
    "DELTA_WT", 
    "DELTA_PRICE", 
    "RETURN",
    "DELTA_VOLUME",
    "PREV_IDX_WT", 
    "PREV_FF_BASED_MCAP",
    # will be created
    "ABS_DELTA_WT",
    "RET_5", 
    "RET_20",
    "VOLAT_5", 
    "VOLAT_20",
    "VOL_SHOCK_5",
    "DAYS_TO_REBAL"
]

# Names of important columns in your sheet:
DATE_COL = "DATE"
SYM_COL = "SYMBOL"
FLOW_COL = "FLOW_PROXY"   # regression target
# Classification target will be created from FLOW_PROXY

In [3]:
# -------------------------
# Utility functions (visuals)
# -------------------------
def plot_feature_heatmap(df, features=FEATURE_COLUMNS):
    corr = df[features].corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=False, cmap="vlag", center=0)
    plt.title("Feature Correlation Heatmap")
    fname = os.path.join(OUTPUT_DIR, "feature_corr_heatmap.png")
    plt.savefig(fname); plt.close()

def plot_time_series_for_symbol(df, symbol, col="FLOW_PROXY", window=60):
    s = df[df[SYM_COL] == symbol].set_index(DATE_COL)[col].dropna()
    if s.empty:
        print(f"No data for {symbol}")
        return
    plt.figure(figsize=(12,4))
    plt.plot(s.index, s.values)
    plt.title(f"{symbol} — {col} (last {window} points)")
    plt.xlabel("Date")
    plt.tight_layout()
    fname = os.path.join(OUTPUT_DIR, f"time_series_{symbol}_{col}.png")
    plt.savefig(fname); plt.close()

def plot_pred_vs_actual(y_true, y_pred, title="Predicted vs Actual"):
    plt.figure(figsize=(12,5))
    plt.plot(y_true, label="Actual", alpha=0.8)
    plt.plot(y_pred, label="Predicted", alpha=0.8)
    plt.legend()
    plt.title(title)
    fname = os.path.join(OUTPUT_DIR, "pred_vs_actual.png")
    plt.savefig(fname); plt.close()

def plot_feature_importance(model, feature_names, top_n=20, fname="feature_importance.png"):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        try:
            importances = model.get_booster().get_score(importance_type='weight')
            # convert dict to vector
            importances = np.array([importances.get(f"f{i}", 0) for i in range(len(feature_names))])
        except Exception:
            print("Model does not expose importances")
            return
    idx = np.argsort(importances)[-top_n:]
    plt.figure(figsize=(8, max(4, len(idx)*0.35)))
    plt.barh(np.array(feature_names)[idx], importances[idx])
    plt.title("Feature Importance (top {})".format(top_n))
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname))
    plt.close()

def plot_confusion(cm, classes, title="Confusion matrix", fname="confusion_matrix.png"):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname))
    plt.close()

def plot_roc(y_true, y_probs, fname="roc_curve.png"):
    from sklearn.metrics import roc_curve, auc
    try:
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(6,5))
        plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
        plt.plot([0,1],[0,1], 'k--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, fname))
        plt.close()
    except Exception as e:
        print("ROC plot failed:", e)

# -------------------------
# Walk-forward validation
# -------------------------
def walk_forward_evaluation(
    df,
    features, #feature columns
    target, #flow column
    date_col=DATE_COL,
    group_col=SYM_COL,
    min_train_days=252*1, # at least ~1 year of trading days to start
    step_days=90, # expand by ~quarter
    model_reg=None, #xgboost regressor
    model_clf=None, #xgboost classifier
    scaler=None #scalar
):
    """
    Expanding window walk-forward:
    - Start at earliest date where we have min_train_days
    - At each step, train on all data up to t_train_end, test on next step_days window
    - Models are trained on pooled cross-section (all symbols), time-aware split by date
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    
    print(df.head())

    dates = df[date_col].drop_duplicates().sort_values().values
    start_index = min_train_days
    # convert to integer indices on unique dates
    results = []
    i = start_index
    while i < len(dates):
        train_end_date = dates[i-1]
        test_end_index = min(i + step_days - 1, len(dates)-1)
        test_start_date = dates[i]
        test_end_date = dates[test_end_index]

        train_df = df[df[date_col] <= train_end_date].dropna(subset=features + [target])
        test_df = df[(df[date_col] >= test_start_date) & (df[date_col] <= test_end_date)].dropna(subset=features + [target])

        if train_df.empty or test_df.empty:
            i += step_days
            continue

        X_train = train_df[features].values
        y_train = train_df[target].values
        X_test = test_df[features].values
        y_test = test_df[target].values

        if scaler:
            scaler.fit(X_train)
            X_train = scaler.transform(X_train)
            X_test = scaler.transform(X_test)

        fold_res = {"train_end": train_end_date, "test_start": test_start_date, "test_end": test_end_date}

        # Regression
        if model_reg is not None:
            model_reg.fit(X_train, y_train)
            preds = model_reg.predict(X_test)
            fold_res.update({
                "reg_r2": r2_score(y_test, preds),
                "reg_mae": mean_absolute_error(y_test, preds),
                "reg_mse": mean_squared_error(y_test, preds)
            })
            # Save a couple of plots for the last fold
            if test_end_index >= len(dates)-1:
                plot_pred_vs_actual(y_test, preds, title=f"Final Fold: Reg Pred vs Actual")
                plot_feature_importance(model_reg, features, top_n=min(20, len(features)), fname="reg_feature_importance.png")
                print('Regression plots saved.')
                # SHAP (optional)
                if _has_shap:
                    explainer = shap.Explainer(model_reg)
                    shap_vals = explainer(X_test)
                    shap.summary_plot(shap_vals, pd.DataFrame(X_test, columns=features), show=False)
                    plt.savefig(os.path.join(OUTPUT_DIR, "reg_shap_summary.png"))
                    plt.close()
                    print('SHAP summary plot saved.')
                    
        # Classification (create labels on the fly)
        if model_clf is not None:
            # --------------------------------------------------------------------
            # DEFINE ALL POSSIBLE CLASSES FOR STABILITY ACROSS FOLDS
            ORIGINAL_CLASSES = np.array([-1, 0, 1])
            MAPPED_CLASSES = np.array([0, 1, 2])
            # Map: -1 (Sell) -> 0, 0 (Neutral) -> 1, 1 (Buy) -> 2
            label_map = {-1: 0, 0: 1, 1: 2}
            reverse_label_map = {v: k for k, v in label_map.items()}
            # --------------------------------------------------------------------
            
            # create labels: 1 = inflow, -1 = outflow, 0 = neutral (tunable threshold)
            thresh = 0.0
            y_train_cls = np.where(y_train > thresh, 1, np.where(y_train < -thresh, -1, 0))
            y_test_cls = np.where(y_test > thresh, 1, np.where(y_test < -thresh, -1, 0))

            # 🚨 NEW STEP: Map custom labels [-1, 0, 1] to XGBoost-compatible [0, 1, 2]
            y_train_cls_mapped = np.array([label_map[y] for y in y_train_cls])
            y_test_cls_mapped = np.array([label_map[y] for y in y_test_cls])
            
            # handle class imbalance with class weights
            # Use the MAPPED classes for weights computation
            classes = np.unique(y_train_cls_mapped)
            if len(classes) > 1:
                # Use the complete set of MAPPED classes for consistency
                cw = class_weight.compute_class_weight("balanced", classes=MAPPED_CLASSES, y=y_train_cls_mapped)
                class_weights_dict = {c: w for c, w in zip(MAPPED_CLASSES, cw)}
                sample_weights = np.array([class_weights_dict[cls] for cls in y_train_cls_mapped])
            else:
                class_weights_dict = None
                sample_weights = None 

            # Fit using the mapped training labels
            model_clf.fit(X_train, y_train_cls_mapped, sample_weight=sample_weights)
            
            # Predict using the mapped labels
            y_pred_cls_mapped = model_clf.predict(X_test)
            
            # Map predictions back to original labels [-1, 0, 1] for metrics
            y_pred_cls = np.array([reverse_label_map[y] for y in y_pred_cls_mapped])
            
            # For ROC/AUC we need prob for positive class (Buy=1); it's in the 3rd column (index 2).
            try:
                # Index 2 of predict_proba corresponds to the mapped label '2', which is 'Buy=1'
                y_prob = model_clf.predict_proba(X_test)[:, 2] 
                print('ROC AUC (Buy=1): {:.4f}'.format(roc_auc_score((y_test_cls==1).astype(int), y_prob)))
            except Exception:
                y_prob = None

            # 🚨 FIX: Use original y_test_cls and y_pred_cls for metrics
            fold_res.update({
                "clf_acc": accuracy_score(y_test_cls, y_pred_cls),
                "clf_f1_macro": f1_score(y_test_cls, y_pred_cls, average="macro"),
                "clf_report": classification_report(y_test_cls, y_pred_cls, labels=ORIGINAL_CLASSES, output_dict=True)
            })
            print('Classification report:\n', classification_report(y_test_cls, y_pred_cls, labels=ORIGINAL_CLASSES))

            if test_end_index >= len(dates)-1:
                # Use original labels for confusion matrix and ROC curve plotting
                cm = confusion_matrix(y_test_cls, y_pred_cls, labels=[-1, 0, 1])
                plot_confusion(cm, classes=["Sell(-1)","Neutral(0)","Buy(1)"], fname="final_confusion_matrix.png")
                
                # ROC curve plots the True Positive Rate vs False Positive Rate for class 'Buy=1'
                if y_prob is not None:
                    plot_roc((y_test_cls==1).astype(int), y_prob, fname="final_roc.png")
            
        results.append(fold_res)
        i += step_days

    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(OUTPUT_DIR, "walk_forward_results.csv"), index=False)
    print("Walk-forward done. Summary saved to:", os.path.join(OUTPUT_DIR, "walk_forward_results.csv"))
    return results_df

In [4]:
# Load
df = pd.read_excel(INPUT_FILE)

# Clean column names
df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")

# CORRECTED code for Excel Date Format
df[DATE_COL] = pd.to_datetime(df[DATE_COL], unit='D', origin='1899-12-30')

df.head()

,DATE,SYMBOL,COMPANY,PRICE,IDX_WT_%,FF_BASED_SHARES,FF_BASED_MCAP,ORD_SHARES,ORD_SHARES_MCAP,VOLUME,PREV_IDX_WT,DELTA_WT,PREV_FF_BASED_MCAP,DELTA_PRICE,DELTA_VOLUME,RETURN,FLOW_PROXY
0,2024-12-23,AIRLINK,Air Link Communication Limited,215.53,0.869499,98817308,2.129809e+10,98817308,2.129809e+10,10266206,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-12-24,AIRLINK,Air Link Communication Limited,210.38,0.861528,98817308,2.078919e+10,98817308,2.078919e+10,8765161,0.869499,-0.007971,2.129809e+10,-5.15,-1501045.0,-0.023895,-1.657077e+08
2,2024-12-26,AIRLINK,Air Link Communication Limited,213.91,0.894374,98817308,2.113801e+10,98817308,2.113801e+10,3834991,0.861528,0.032846,2.078919e+10,3.53,-4930170.0,0.016779,6.942939e+08
3,2024-12-27,AIRLINK,Air Link Communication Limited,215.74,0.895306,98817308,2.131885e+10,98817308,2.131885e+10,3848039,0.894374,0.000932,2.113801e+10,1.83,13048.0,0.008555,1.987233e+07
4,2024-12-30,AIRLINK,Air Link Communication Limited,217.38,0.870778,98817308,2.148091e+10,98817308,2.148091e+10,1497575,0.895306,-0.024529,2.131885e+10,1.64,-2350464.0,0.007602,-5.268953e+08


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42330 entries, 0 to 42329
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype        
---  ------              --------------  -----        
 0   DATE                42330 non-null  datetime64[s]
 1   SYMBOL              42330 non-null  str          
 2   COMPANY             42330 non-null  str          
 3   PRICE               42330 non-null  float64      
 4   IDX_WT_%            42330 non-null  float64      
 5   FF_BASED_SHARES     42330 non-null  int64        
 6   FF_BASED_MCAP       42330 non-null  float64      
 7   ORD_SHARES          42330 non-null  int64        
 8   ORD_SHARES_MCAP     42330 non-null  float64      
 9   VOLUME              42330 non-null  int64        
 10  PREV_IDX_WT         42271 non-null  float64      
 11  DELTA_WT            42271 non-null  float64      
 12  PREV_FF_BASED_MCAP  42271 non-null  float64      
 13  DELTA_PRICE         42271 non-null  float64      
 14  DELTA_VOLUME     

In [6]:
df["RET_5"] = df.groupby("SYMBOL")["PRICE"].pct_change(5)
df["RET_20"] = df.groupby("SYMBOL")["PRICE"].pct_change(20)

df["VOLAT_5"] = df.groupby("SYMBOL")["PRICE"].rolling(5).std().reset_index(0,drop=True)
df["VOLAT_20"] = df.groupby("SYMBOL")["PRICE"].rolling(20).std().reset_index(0,drop=True)

df["VOL_SHOCK_5"] = df.groupby("SYMBOL")["VOLUME"].pct_change(5)

df["ABS_DELTA_WT"] = df["DELTA_WT"].abs()

rebalancing_dates = [
    "2020-03-15",
    "2020-09-15",
    "2021-03-15",
    "2021-09-15",
    "2022-03-15",
    "2022-09-15",
    "2023-03-15",
    "2023-09-15",
    "2024-03-15",
    "2024-09-15",
    "2025-03-15",
    "2025-09-15",
]
# OPTIONAL: Convert once for speed
rebalancing_timestamps = [pd.Timestamp(d) for d in rebalancing_dates] 

df["DAYS_TO_REBAL"] = df["DATE"].apply(
    lambda x: min(
        [(d_ts - x).days for d_ts in rebalancing_timestamps if d_ts > x]
    ) 
    if any(d_ts > x for d_ts in rebalancing_timestamps) 
    else np.nan 
)

df.head()

,DATE,SYMBOL,COMPANY,PRICE,IDX_WT_%,FF_BASED_SHARES,FF_BASED_MCAP,ORD_SHARES,ORD_SHARES_MCAP,VOLUME,...,DELTA_VOLUME,RETURN,FLOW_PROXY,RET_5,RET_20,VOLAT_5,VOLAT_20,VOL_SHOCK_5,ABS_DELTA_WT,DAYS_TO_REBAL
0,2024-12-23,AIRLINK,Air Link Communication Limited,215.53,0.869499,98817308,2.129809e+10,98817308,2.129809e+10,10266206,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.0
1,2024-12-24,AIRLINK,Air Link Communication Limited,210.38,0.861528,98817308,2.078919e+10,98817308,2.078919e+10,8765161,...,-1501045.0,-0.023895,-1.657077e+08,NaN,NaN,NaN,NaN,NaN,0.007971,81.0
2,2024-12-26,AIRLINK,Air Link Communication Limited,213.91,0.894374,98817308,2.113801e+10,98817308,2.113801e+10,3834991,...,-4930170.0,0.016779,6.942939e+08,NaN,NaN,NaN,NaN,NaN,0.032846,79.0
3,2024-12-27,AIRLINK,Air Link Communication Limited,215.74,0.895306,98817308,2.131885e+10,98817308,2.131885e+10,3848039,...,13048.0,0.008555,1.987233e+07,NaN,NaN,NaN,NaN,NaN,0.000932,78.0
4,2024-12-30,AIRLINK,Air Link Communication Limited,217.38,0.870778,98817308,2.148091e+10,98817308,2.148091e+10,1497575,...,-2350464.0,0.007602,-5.268953e+08,NaN,NaN,2.654086,NaN,NaN,0.024529,75.0


In [7]:
# Rename constants if needed (align with above)
global DATE_COL, SYM_COL, FLOW_COL, FEATURE_COLUMNS
DATE_COL = "DATE"
SYM_COL = "SYMBOL"
FLOW_COL = "FLOW_PROXY"

# --- 1. SORT: Ensure data is sorted by SYMBOL and DATE ---
df = df.sort_values([SYM_COL, DATE_COL])

# --- 2. Convert Inf to NaN ---
# This step handles cases where pct_change() produced Inf (e.g., division by zero volume)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print("Replaced Inf values with NaN.")

# Drop rows where we don't have target OR any of the features
df = df.dropna(subset=[FLOW_COL] + FEATURE_COLUMNS)

print(f"Dataframe cleaned. Final shape: {df.shape}")

Replaced Inf values with NaN.
Dataframe cleaned. Final shape: (40771, 24)


In [8]:
df.info()

<class 'pandas.DataFrame'>
Index: 40771 entries, 20 to 42329
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype        
---  ------              --------------  -----        
 0   DATE                40771 non-null  datetime64[s]
 1   SYMBOL              40771 non-null  str          
 2   COMPANY             40771 non-null  str          
 3   PRICE               40771 non-null  float64      
 4   IDX_WT_%            40771 non-null  float64      
 5   FF_BASED_SHARES     40771 non-null  int64        
 6   FF_BASED_MCAP       40771 non-null  float64      
 7   ORD_SHARES          40771 non-null  int64        
 8   ORD_SHARES_MCAP     40771 non-null  float64      
 9   VOLUME              40771 non-null  int64        
 10  PREV_IDX_WT         40771 non-null  float64      
 11  DELTA_WT            40771 non-null  float64      
 12  PREV_FF_BASED_MCAP  40771 non-null  float64      
 13  DELTA_PRICE         40771 non-null  float64      
 14  DELTA_VOLUME        4

In [9]:
# Quick EDA visuals
print("Creating feature heatmap...")
plot_feature_heatmap(df, features=FEATURE_COLUMNS)
print("Plotting some example time series for first 5 symbols...")
# for sym in df[SYM_COL].unique()[:5]:
#     plot_time_series_for_symbol(df, sym, col=FLOW_COL)

Creating feature heatmap...
Plotting some example time series for first 5 symbols...


In [10]:
# Prepare models
xgb_reg = XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
)
xgb_clf = XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
    eval_metric="logloss", random_state=42
)
scaler = StandardScaler()

In [11]:
# Run walk-forward evaluation
results_df = walk_forward_evaluation(
    df=df,
    features=FEATURE_COLUMNS,
    target=FLOW_COL,
    date_col=DATE_COL,
    model_reg=xgb_reg,
    model_clf=xgb_clf,
    scaler=scaler,
    min_train_days=252,   # start after ~1 year of data
    step_days=63          # test on next quarter ~63 trading days
)
print(results_df.describe(include='all'))
print("All done. Check the fyp_outputs folder for figures and CSVs.")

            DATE SYMBOL                          COMPANY   PRICE  IDX_WT_%  \
29065 2020-01-29   PAEL               Pak Elektron Ltd.    27.25      0.51   
2173  2020-01-29   BAFL               Bank Alfalah Ltd.    51.90      2.78   
9636  2020-01-29   EPCL  Engro Polymer & Chemicals Ltd.    32.70      0.78   
2844  2020-01-29   BAHL              Bank Al-Habib Ltd.    81.57      4.44   
33683 2020-01-29    PSO      Pakistan State Oil Co Ltd.   210.42      3.35   

       FF_BASED_SHARES  FF_BASED_MCAP  ORD_SHARES  ORD_SHARES_MCAP   VOLUME  \
29065        248840743   6.780910e+09   497681485     1.356182e+10  7048000   
2173         710866048   3.689395e+10  1777165119     9.223487e+10   218000   
9636         318123167   1.040263e+10   908923333     2.972179e+10  1124500   
2844         722426520   5.892833e+10  1111425419     9.065897e+10   740000   
33683        211262985   4.445396e+10   469473300     9.878657e+10  1679800   

       ...  DELTA_VOLUME    RETURN    FLOW_PROXY     RET

ExplainerError: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. Consider retrying with the feature_perturbation='interventional' option. This check failed because for one of the samples the sum of the SHAP values was -74865180672.000000, while the model output was -74865065984.000000. If this difference is acceptable you can set check_additivity=False to disable this check.